In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np
import pandas as pd

from src.data import load_etth1, make_dataset, make_windows
from src.metrics import mse, sse
from src.solvers import fit_with_bias, nlinear_constrained, ols, ridge

In [2]:
x, y = make_windows(np.arange(10), 3, 2)
print(x.shape, y.shape)

(6, 3) (6, 2)


In [3]:
rng = np.random.default_rng(0)
W_true = rng.normal(size=(2, 3))
X_toy = rng.normal(size=(100, 3))
Y_toy = X_toy @ W_true.T          # dữ liệu sinh đúng từ W_true, không nhiễu

W_hat = ols(X_toy, Y_toy)
print(np.abs(W_hat - W_true).max())

2.220446049250313e-16


In [4]:
np.abs(ridge(X_toy, Y_toy, 0) - ols(X_toy, Y_toy)).max()

np.float64(0.0)

In [5]:
nlinear_constrained(X_toy, Y_toy)

array([[ 0.22505322, -0.00703769,  0.78198446],
       [ 0.3950846 , -0.17027008,  0.77518548]])

In [6]:
# Ràng buộc tổng hàng = 1 chỉ có thể làm MSE train tăng (W_true không thoả ràng buộc)
W_ols = ols(X_toy, Y_toy)
V     = nlinear_constrained(X_toy, Y_toy)
print(mse(W_ols, 0, X_toy, Y_toy))
print(mse(V,     0, X_toy, Y_toy))

2.346891586614445e-32
0.24222626009943105


In [7]:
b_true = rng.normal(size=2)
Y_b = Y_toy + b_true
W, b = fit_with_bias(ols, X_toy, Y_b)   # truyền ols, không có ngoặc
print(np.abs(W - W_true).max(), np.abs(b - b_true).max())

1.1102230246251565e-16 2.220446049250313e-16


In [8]:
L, H = 336, 96
train, val, test = load_etth1("../data/ETTh1.csv", L)
X_train, Y_train = make_dataset(train, L, H)
X_test,  Y_test  = make_dataset(test,  L, H)
print(X_train.shape, X_test.shape)

W, b = fit_with_bias(ols, X_train, Y_train)
pred =   X_test @ W.T + b                 
print(np.mean((Y_test - pred) ** 2))


(57463, 336)

 (19495, 336)


0.37023527067813916


In [9]:
W_ols, b_ols = fit_with_bias(ols, X_train, Y_train)
V,     b_V   = fit_with_bias(nlinear_constrained, X_train, Y_train)
lhs = sse(V, b_V, X_train, Y_train) - sse(W_ols, b_ols, X_train, Y_train)  # TODO: hiệu SSE train, NLinear trừ OLS

# --- 2. Vế phải: chỉ từ định nghĩa, KHÔNG dùng V hay b_V ---
X_c = X_train - X_train.mean(axis = 0)   # TODO: X_train trừ trung bình cột
A   =( X_c.T @ X_c) # TODO
L = np.ones(336)
u = np.linalg.solve(X_c.T @ X_c, L)
s   = np.ones(X_train.shape[1]).T @ u   # TODO: s = 1_L^T u
r   = W_ols@np.ones(X_c.shape[1]) - np.ones(Y_train.shape[1])    # TODO: r = W_ols 1_L − 1_H
rhs =  np.linalg.norm(r)**2 /s   # TODO: ||r||^2 / s


print(lhs, rhs, abs(lhs - rhs) / rhs)

print(lhs) 
print(u.shape)

17002.829247226007 17002.82924722584 9.842304636066549e-15
17002.829247226007
(336,)


In [10]:
lhs/Y_train.size

np.float64(0.0030822060222857185)

In [11]:
print(sse(V, b_V,X_test, Y_test)/Y_test.size)

0.3701986328402697


In [12]:
# Độ lệch tổng hàng khỏi 1: OLS vi phạm ràng buộc, V thoả tới sai số máy
print(np.abs(W_ols.sum(axis=1) - 1).max())
print(np.abs(V.sum(axis=1) - 1).max())

0.1083396911242277
4.440892098500626e-16
